In [ ]:
# Q1. Embedding a query

# Embed the following query:

#     How does approximate nearest neighbor search work?

# The embedder returns a vector of 384 numbers. What's the first value (v[0])?

In [ ]:

from embedder import Embedder

model = Embedder()

query = "How does approximate nearest neighbor search work?"
v = model.encode(query)

print(v[0])

-0.02058203437252893


In [ ]:
# Q2. Cosine similarity

# The embedder returns normalized vectors, so the dot product between two of them is their cosine similarity.

# Take the page 02-vector-search/lessons/07-sqlitesearch-vector.md, embed its content, and compute the cosine similarity with the query vector from Q1. What do you get?

In [10]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [11]:
doc = next(
    d for d in documents
    if d["filename"] == "02-vector-search/lessons/07-sqlitesearch-vector.md"
)

In [12]:
doc_vector = model.encode(doc["content"])

In [13]:
query = "How does approximate nearest neighbor search work?"
query_vector = model.encode(query)

In [14]:
import numpy as np

similarity = np.dot(query_vector, doc_vector)
print(similarity)

0.36107027225589694


In [ ]:
# Q3. Chunking and search by hand

# A full page covers several topics, which waters down its embedding.

# We chunk the pages the same way as in homework 1:

In [15]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

In [16]:
texts = [chunk["content"] for chunk in chunks]
X = model.encode_batch(texts)

In [18]:
# Score against query, higher score = more relevant chunk  
scores = X.dot(v)
print(scores)

[ 3.15187611e-01  2.01479664e-01  5.90559037e-02 -7.67661288e-02
  1.18452466e-01 -1.41697012e-01 -2.81406495e-02 -4.65669117e-02
 -2.06994543e-02 -6.07744147e-02  2.13273769e-01  8.87600958e-02
 -1.97269268e-02  3.11629985e-01  5.51034635e-01  2.04008152e-01
  2.12515802e-01  1.93649107e-01  2.51961267e-01  1.31078610e-01
  2.59120607e-01  7.63816369e-02  9.59193203e-02  9.81471228e-03
 -3.59107168e-02  1.01211406e-02  1.10786940e-01 -9.90259915e-02
 -3.71170645e-02  7.59057333e-02 -3.35340234e-02  8.86841484e-03
  1.02636448e-01  6.89615413e-02  1.29408854e-01  2.57709121e-01
  3.23680576e-01  1.06350076e-01  5.61891208e-02  2.34017441e-01
  1.97954358e-01  9.64296342e-02  1.93709934e-01  2.16719269e-01
  3.48340450e-01  5.10906541e-02  2.05212833e-01  1.05416182e-01
 -3.25432660e-02  4.94665347e-02  2.38574873e-01 -3.44207304e-02
  1.82165430e-01  2.77929421e-02 -2.26806900e-03  2.91897549e-01
  4.39332202e-02  2.23759662e-01  1.63523531e-01  9.16125960e-02
 -8.39985178e-02  7.88591

In [19]:
# Find best chunk
best_idx = scores.argmax()
chunks[best_idx]["filename"]

'02-vector-search/lessons/07-sqlitesearch-vector.md'

In [ ]:
# Q4. Vector search with minsearch

# We've done vector search by hand, which is good for learning, but it's not what we do in practice. In practice we use libraries.

# Let's use VectorSearch from minsearch and run a search for the following query:

#     What metric do we use to evaluate a search engine?

# Which file is the filename of the first result?

#     02-vector-search/lessons/04-vector-search.md
#     04-evaluation/lessons/05-search-metrics.md
#     04-evaluation/lessons/13-llm-as-judge.md
#     05-monitoring/lessons/04-metrics.md


In [20]:
from minsearch import VectorSearch

In [25]:
index = VectorSearch()
index.fit(X, chunks)

In [30]:
query = "What metric do we use to evaluate a search engine?"

# 1. Encode the text query into a vector first
query_vector = model.encode(query)

# 2. Pass the encoded vector to the search method
results = index.search(
    query_vector,
    num_results=5
)

# You can then check the filename of the first result
print(results[0]["filename"])


04-evaluation/lessons/05-search-metrics.md


In [ ]:
# Q5. Text search vs vector search

# Vector search matches by meaning, keyword search by exact words.

# Let's compare them. Index the same chunks with Index from minsearch. Use content as a text field.

# Run both searches for this query:

#     How do I store vectors in PostgreSQL?


In [ ]:
# Text Search
from minsearch import Index

text_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

text_index.fit(chunks)

query = "How do I store vectors in PostgreSQL?"

text_results = text_index.search(
    query=query,
    filter_dict={},
    num_results=5
)

for r in text_results:
    print(r["filename"])

02-vector-search/lessons/02-embeddings.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/01-intro.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/01-intro.md


In [39]:
from minsearch import VectorSearch

vector_index = VectorSearch()

vector_index.fit(X, chunks)

query_vector =  model.encode(query)

vector_results = vector_index.search(
    query_vector,
    num_results=5
)
for r in vector_results:
    print(r["filename"])

02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md


In [41]:
# Compare Results.. 
text_files = {r["filename"] for r in text_results}
vector_files = {r["filename"] for r in vector_results}

print(vector_files - text_files)

{'02-vector-search/lessons/08-pgvector.md'}


In [ ]:
# Q6 — What concept is being tested?

# This tests Hybrid Search = Vector Search + Text Search + Rank Fusion

# Why hybrid?

# Because each method fails differently.

# Vector Search strengths

# Good at meaning.

In [ ]:
# Q6. Hybrid search

# Both vector and text search have their strengths and weaknesses. Vector search matches by meaning, so it finds relevant pages even when they use words different from the query. But it can miss exact terms like names, codes, or rare keywords. Text search is the opposite: it nails exact words but misses paraphrases and synonyms.

# We don't have to pick one or the other - we can use both and merge their results. This approach is called "hybrid search".

# Each search produces its own ranked list, so we need a way to combine them into one. In this homework we use Reciprocal Rank Fusion (RRF). It ignores the raw scores from each method, which live on different scales and aren't directly comparable. Instead, it looks only at the position of each document in each list.

# Every document scores by its position (rank, starting at 0) in each list, and we sum the scores across lists with a constant k = 60:

# RRF(d) = sum over lists of  1 / (k + rank(d))

# "Sum over lists" means we go through every ranked list and, for each list where the document appears, add its 1 / (k + rank) contribution. A document found by both searches collects a score from each list, while one found by only a single search collects just one.

# The constant k controls how much the exact rank matters. A larger k flattens the gap between positions, so the difference between rank 0 and rank 5 counts for less. A smaller k does the opposite: it sharpens that gap, so being at the top of a list matters much more.

# The value 60 comes from the original RRF paper and is the usual default. You rarely need to tune it. Lower it when only the top results matter. Raise it to reward documents that appear across many lists, even when they never quite reach the top.

# A document that ranks well in both lists ends up higher than one that's only strong in a single list.

In [44]:
query = "How do I give the model access to tools?"
query_vector=model.encode(query)

In [46]:
vector_results = vector_index.search(
    query_vector,
    num_results=5
)

In [47]:
for r in vector_results:
    print(r["filename"], r["start"])

01-agentic-rag/lessons/01-intro.md 2000
04-evaluation/lessons/02-ground-truth.md 1000
01-agentic-rag/lessons/16-other-frameworks.md 0
01-agentic-rag/lessons/15-frameworks.md 2000
01-agentic-rag/lessons/13-function-calling.md 4000


In [48]:
text_results = text_index.search(
    query=query,
    filter_dict={},
    num_results=5
)

In [49]:
for r in text_results:
    print(r["filename"], r["start"])

01-agentic-rag/lessons/14-agentic-loop.md 0
01-agentic-rag/lessons/13-function-calling.md 4000
01-agentic-rag/lessons/13-function-calling.md 5000
01-agentic-rag/lessons/13-function-calling.md 1000
04-evaluation/lessons/02-ground-truth.md 3000


In [50]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])

            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [51]:
results = rrf([vector_results, text_results])

In [52]:
print(results[0]["filename"])

01-agentic-rag/lessons/13-function-calling.md


In [53]:
for r in results:
    print(r["filename"], r["start"])

01-agentic-rag/lessons/13-function-calling.md 4000
01-agentic-rag/lessons/01-intro.md 2000
01-agentic-rag/lessons/14-agentic-loop.md 0
04-evaluation/lessons/02-ground-truth.md 1000
01-agentic-rag/lessons/16-other-frameworks.md 0
